## 1. ECG 切片

In [2]:
import os, re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import neurokit2 as nk

data_path = './data_parquets/'
gb_name = data_path + 'gb_chunks_2026-02-23_001806.parquet'
ecg_time_str = re.search(r'\d{4}-\d{2}-\d{2}_\d{6}', gb_name).group()
chunks = pd.read_parquet(gb_name)

## 2. 画出所有 Bad 样本

In [3]:
from scipy.signal import find_peaks
class BeatCalcFS:
    def __init__(self, df, ecg, fs):
        self.ecg = df[ecg]
        self.fs = fs
        distance = int(self.fs * 60 / 200)
        self.peaks, _ = find_peaks(self.ecg, height=450, distance=distance)
        self.rr_intervals = np.diff(self.peaks)/self.fs

In [ ]:
def save_chunk_plot(chunks, idx, label, base_dir=f"./ecg_figs/{ecg_time_str}", fs=250):
    # 建目录：ecg_time_str/good, ecg_time_str/bad, ecg_time_str/bad2good
    save_dir = os.path.join(base_dir, label)
    os.makedirs(save_dir, exist_ok=True)

    ecg = pd.DataFrame(chunks.loc[idx, "ecg_raw"])
    ecg.columns = ['ecg_raw']
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 4))

    ax.plot(ecg['ecg_raw'], alpha=0.4, label="Raw")
    my_r_peaks = BeatCalcFS(ecg, 'ecg_raw', 250).peaks
    ax.scatter(my_r_peaks,
                ecg.loc[my_r_peaks],
                color="C0", s=20)

    try:
        ecg_signals, _ = nk.ecg_process(ecg['ecg_raw'], sampling_rate=fs)
        ax.plot(ecg_signals["ECG_Clean"], label="Clean")
        nk_r_peaks = ecg_signals.index[ecg_signals["ECG_R_Peaks"] != 0]
        ax.scatter(nk_r_peaks,
                ecg_signals.loc[nk_r_peaks, "ECG_Clean"],
                color="r", s=20)
    except Exception as e:
        print(f"警告：Chunk {i} 烂到无法处理，仅绘制原始波形。错误原因: {e}")
        # 仅绘制 Raw 数据，不依赖 NK2 的处理结果


    ax.set_title(f"Chunk {idx} | {label}")
    ax.set_xlabel("Samples")
    ax.set_ylabel("Amplitude")
    ax.legend(loc="upper right")

    filename = f"{idx}_chunk_{label}.jpg"
    path = os.path.join(save_dir, filename)

    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.close(fig)

for i in chunks.index:
    if chunks.loc[i, 'label'] == 'bad':
        save_chunk_plot(chunks, i, 'bad')

d:\Softwares\Miniconda\envs\ecg_env\lib\site-packages\neurokit2\signal\signal_period.py:84: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(


警告：Chunk 1061 烂到无法处理，仅绘制原始波形。错误原因: cannot convert float NaN to integer


d:\Softwares\Miniconda\envs\ecg_env\lib\site-packages\neurokit2\signal\signal_period.py:84: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(


警告：Chunk 1142 烂到无法处理，仅绘制原始波形。错误原因: cannot convert float NaN to integer


d:\Softwares\Miniconda\envs\ecg_env\lib\site-packages\neurokit2\signal\signal_period.py:84: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(


警告：Chunk 1143 烂到无法处理，仅绘制原始波形。错误原因: cannot convert float NaN to integer


d:\Softwares\Miniconda\envs\ecg_env\lib\site-packages\neurokit2\signal\signal_period.py:84: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(


警告：Chunk 1155 烂到无法处理，仅绘制原始波形。错误原因: cannot convert float NaN to integer
